In [ ]:
# https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data"

df = pd.read_csv(url,header=None)
df.head()


In [ ]:
print(df.shape)

In [ ]:
df.columns

In [ ]:
column_names = [
    'symboling', 'normalized-losses', 'make', 'fuel-type', 'aspiration',
    'num-of-doors', 'body-style', 'drive-wheels', 'engine-location',
    'wheel-base', 'length', 'width', 'height', 'curb-weight', 'engine-type',
    'num-of-cylinders', 'engine-size', 'fuel-system', 'bore', 'stroke',
    'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg', 'price'
]
df.columns = column_names
df.head()

In [ ]:
print(df.info())

In [ ]:
print(df.describe())

In [ ]:
df.replace("?",np.nan,inplace=True)

print(df.isnull().sum())

In [ ]:
print(df.dtypes)

In [ ]:
df.head(2)

In [ ]:

numeric_cols = ['normalized-losses','wheel-base','length','height','curb-weight','engine-size','bore','stroke','compression-ratio','horsepower','peak-rpm','city-mpg','highway-mpg','price']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col],errors='coerce')

print(df.dtypes)

In [ ]:
# missing values 

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

df['num-of-doors'] = df['num-of-doors'].fillna(df['num-of-doors'].mode()[0])

print(df.isnull().sum())

In [ ]:
df.head()

In [ ]:
# check duplicates 
print(df.duplicated().sum())

df.to_csv("Auto_Mobile.csv")

In [ ]:
sns.histplot(df['price'],bins=30,kde=True,color='skyblue')
plt.title("Distribution of car price")
plt.xlabel("price")
plt.ylabel("number of car")
plt.show()

In [ ]:
sns.boxplot(y=df['price'],color='lightgreen')
plt.title("box plot of car price outliers ")
plt.xlabel("price")
plt.ylabel("number of car")
plt.show()

In [ ]:
df.columns

In [ ]:
# engine size vs price 
sns.scatterplot(data=df,x='engine-size',y='price',hue="fuel-type",palette='viridis',s=100)
plt.title("Engine vs price")
plt.xlabel("Engine size")
plt.ylabel("price")
plt.legend(title="Fuel Type")
plt.show()

In [ ]:
sns.boxplot(data=df,x="body-style",y='price',palette="Set2",hue='body-style')
plt.title("price distribution by body style")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# count of cars by body style
sns.countplot(data=df,x="body-style",palette='pastel',
              order=df['body-style'].value_counts().index,hue='body-style')
plt.title("Number of cars by body style")
plt.show()

In [ ]:
df.describe()

In [ ]:
# outlier detection and handling 
# IQR method

q1 = df['horsepower'].quantile(0.25)
q3 = df['horsepower'].quantile(0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr


outliers = df[(df['horsepower'] < lower) | ((df['horsepower'] > upper))]
print(outliers[['make','horsepower']])

In [ ]:
# Z score method

from scipy import stats

df['zscore'] = stats.zscore(df['horsepower'])
z_outliers = df[np.abs(df['zscore']) > 3]

print(z_outliers[['make','horsepower']])

In [ ]:
df['num-of-cylinders'].tail()

In [ ]:
# One hot encoding (binary columns) category

df_onehot = pd.get_dummies(df,columns=['drive-wheels'],prefix='drive')
print(df_onehot.head())


In [ ]:
# label Encoding

cyl_map = {'four': 4,'five':5,'six':6}

df['num-of-cylinders_encoded'] = df['num-of-cylinders'].map(cyl_map)
print(df[['num-of-cylinders','num-of-cylinders_encoded']])

In [ ]:
df.columns

In [ ]:
unique = pd.unique(df['price'])
print(unique)
print(len(unique))

In [ ]:
# Data Scaling (Normalization)/min-max standardization

# min-max standardization [0,1]
df['wheel-base_min_max'] = (df['wheel-base'] - df['wheel-base'].min()) / (df['wheel-base'].max() - df['wheel-base'].min())

print(df['wheel-base_min_max'])

In [ ]:
# standardization (mean=0 , std =1)
df['wheel-base_std'] = (df['wheel-base'] - df['wheel-base'].mean()) / df['wheel-base'].std()
print(df['wheel-base_std'])

In [ ]:
# feature selection and feature engineering

# 1 feature selection(correlation)

corr_matrix = df.corr(numeric_only=True)

print(corr_matrix['price'].sort_values(ascending=False))

In [ ]:
# Average MPG 
df['avg-mpg'] = (df['city-mpg'] + df['highway-mpg'])/2
print(df['avg-mpg'])

# curb-weight + horsepower / 2 = power of weight

In [ ]:
# handling imbalanced data (SMOTE , Oversampling)

mean_price = df['price'].mean()
df['high_price'] = np.where(df['price'] > mean_price , 1 , 0)
print(df['high_price'].value_counts())

In [ ]:
# reduce  majority class (under sampling)
majority = df[df['high_price'] == 0]
min_majority = df[df['high_price'] == 1]

majority_under = majority.sample(len(min_majority))
df_under = pd.concat([majority_under ,min_majority],axis=0)
print(df_under['high_price'].value_counts())


In [ ]:
# oversampling
majority_over = min_majority.sample(len(majority),replace=True)
df_over = pd.concat([majority ,majority_over],axis=0)
print(df_over['high_price'].value_counts())